1. Run: `python -m pip install pydantic fastapi "uvicorn[standard]"`

## Program 1 — A normal Python dictionary

A dictionary can hold a product's details, but Python does not automatically check whether the values make sense.

In [ ]:
# PROGRAM 1: A product stored in a normal dictionary

# A dictionary stores information as key-value pairs.
product = {
    "name": "Keyboard",
    "price": 1200
}

# Read values from the dictionary using their keys.
print("Product name:", product["name"])
print("Product price:", product["price"])

# Try changing 1200 to "expensive".
# The dictionary will still accept it. A normal dictionary does not
# automatically check that price is a number.
product["price"] = "expensive"
print("Price after incorrect change:", product["price"])

## Program 2 — Create your first Pydantic model

A Pydantic model is a set of rules for the shape of some data.

- `BaseModel` gives the model Pydantic's validation behavior.
- `name: str` says the name should be text.
- `price: float` says the price should be a number.

Run this program with valid data first. Then try changing the price to `"expensive"` and run it again. You should see a validation error.

In [ ]:
# PROGRAM 2: Validate data with a basic Pydantic model

from pydantic import BaseModel

# Define a model named Product.
# It expects every product to have a name and a price.
class Product(BaseModel):
    name: str       # Expected type: text (string)
    price: float    # Expected type: number (float)


# Create an object that follows the Product rules.
product = Product(name="Keyboard", price=1200)

# Access model values with a dot, just like product.name.
print("Name:", product.name)
print("Price:", product.price)
print("Python type of price:", type(product.price))

# Experiment:
# Change price=1200 to price="expensive".
# Pydantic will reject the value because it cannot convert it to a number.

## Program 3 — Missing required information

In Program 2, both `name` and `price` are required. Try removing `price=1200` from the line that creates the product. Pydantic will report that the required field is missing.

In [ ]:
# PROGRAM 3: Required fields

from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: float  # Required because no default value is provided.


# This works because both required fields are present.
product = Product(name="Mouse", price=500)
print(product)

# Experiment:
# Uncomment the next line and run the program.
# It will fail because price is missing.
# product_without_price = Product(name="Mouse")


## Program 4 — Optional fields and default values

Sometimes information is not mandatory. Here, `description` is allowed to be missing.

Read `description: str | None = None` like this:
- `str | None`: the value may be text or `None`.
- `= None`: if the caller leaves it out, use `None` by default.

The `name` and `price` fields are still required.

In [ ]:
# PROGRAM 4: An optional description

from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: float
    description: str | None = None  # Optional; defaults to None.


# Description is omitted, so it uses the default None.
product_one = Product(name="Mouse", price=500)
print("First product:", product_one)

# Description is provided, so it contains the supplied text.
product_two = Product(
    name="Keyboard",
    price=1200,
    description="Wireless keyboard"
)
print("Second product:", product_two)
print("Second description:", product_two.description)


## Program 5 — Add simple validation rules with Field

Types alone are not enough. A price could be `-10`, which is a number but usually not a valid product price.

`Field(gt=0)` means the price must be greater than zero.
`Field(min_length=2, max_length=50)` limits the length of the product name.

Try the valid version first. Then change the price to `-10` or the name to `"A"` and run again to see the validation error.

In [ ]:
# PROGRAM 5: Validate the actual values, not just their types

from pydantic import BaseModel, Field

class Product(BaseModel):
    # The name must contain at least 2 and at most 50 characters.
    name: str = Field(min_length=2, max_length=50)

    # gt=0 means "greater than zero".
    price: float = Field(gt=0)


# Valid data: the name is long enough and price is positive.
product = Product(name="Keyboard", price=1200)
print(product)

# Experiments:
# 1. Change price=1200 to price=-10.
# 2. Change name="Keyboard" to name="A".
# Each change should cause a Pydantic validation error.


## Program 6 — Handle a validation error without crashing the whole script

Pydantic raises `ValidationError` when the data breaks the model rules. We can catch that error and print a readable explanation.

This program deliberately tries an invalid price, so it should finish normally and print the validation details.

In [ ]:
# PROGRAM 6: Catch and display a validation error

from pydantic import BaseModel, Field, ValidationError

class Product(BaseModel):
    name: str = Field(min_length=2)
    price: float = Field(gt=0)


try:
    # This is invalid because price must be greater than zero.
    product = Product(name="Keyboard", price=-10)
    print("Product created:", product)

except ValidationError as error:
    # Catch the validation problem instead of letting it stop the script.
    print("The product data is invalid.")
    print(error)


## Program 7 — See how Pydantic handles compatible types

Pydantic can convert some compatible values. For example, the text `"1200"` can usually be converted to a number when the field expects a float. But `"expensive"` cannot be converted to a number.

This is called parsing/coercion. If an API must reject values that arrive as the wrong type, stricter validation can be configured.

In [ ]:
# PROGRAM 7: Compatible input can be converted

from pydantic import BaseModel, ValidationError

class Product(BaseModel):
    name: str
    price: float


# The string "1200" can be converted to the numeric value 1200.0.
product = Product(name="Keyboard", price="1200")
print("Converted price:", product.price)
print("Converted price type:", type(product.price))

# This cannot be converted to a number, so validation fails.
try:
    Product(name="Keyboard", price="expensive")
except ValidationError as error:
    print("\nInvalid price was rejected:")
    print(error)


## Program 8 — Convert a Pydantic model back into a dictionary

Sometimes you need ordinary dictionary data, for example to prepare a response or pass data to another function.

In Pydantic v2, use `model_dump()` to get a dictionary of the model's fields.

In [ ]:
# PROGRAM 8: Turn a model into a dictionary

from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: float
    description: str | None = None


product = Product(
    name="Keyboard",
    price=1200,
    description="Wireless"
)

# model_dump() returns the fields as a normal Python dictionary.
product_dictionary = product.model_dump()

print("Model:", product)
print("Dictionary:", product_dictionary)
print("Dictionary type:", type(product_dictionary))


## Program 9 — Use Pydantic in a FastAPI POST endpoint

Now connect the ideas to an API.

What happens:
1. A client sends JSON to `POST /products`.
2. FastAPI reads the request body.
3. Because the function parameter is `product: ProductCreate`, FastAPI asks Pydantic to validate the body.
4. If valid, the function runs and returns a response.
5. If invalid, FastAPI returns a validation error (normally HTTP 422).

Save this entire cell as `api_program_9.py`. Run it from the terminal with:
`uvicorn api_program_9:app --reload`

Then open `http://127.0.0.1:8000/docs` in a browser and try the endpoint.

This is a separate program; it needs FastAPI and Uvicorn installed.

In [ ]:
# PROGRAM 9: A small FastAPI endpoint that uses Pydantic

from fastapi import FastAPI
from pydantic import BaseModel, Field

# Create the FastAPI application.
app = FastAPI()


# This model describes the data the client must send.
class ProductCreate(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    price: float = Field(gt=0)


# POST means the client is sending data to create something.
@app.post("/products")
def create_product(product: ProductCreate):
    # FastAPI validates the JSON body before this function runs.
    # We can access validated fields using dot notation.
    return {
        "message": "Product data accepted",
        "name": product.name,
        "price": product.price
    }


# To run this program:
# uvicorn api_program_9:app --reload
#
# In /docs, try this valid JSON:
# {
#   "name": "Keyboard",
#   "price": 1200
# }
#
# Then try this invalid JSON:
# {
#   "name": "A",
#   "price": -10
# }


## Program 10 — Separate the incoming data from the response shape

A model used to receive data does not have to be identical to a model used to return data.

Here:
- `ProductCreate` describes what the client must send.
- `ProductResponse` describes what the API returns.
- The response includes an `id` generated by the application, so the client does not need to send it.

This example uses an in-memory counter, not a database. Save it as `api_program_10.py` and run:
`uvicorn api_program_10:app --reload`

In [ ]:
# PROGRAM 10: A request model and a response model

from fastapi import FastAPI, status
from pydantic import BaseModel, Field

app = FastAPI()

class ProductCreate(BaseModel):
    # These fields must be supplied by the client.
    name: str = Field(min_length=2)
    price: float = Field(gt=0)


class ProductResponse(BaseModel):
    # The API returns these fields after accepting the request.
    id: int
    name: str
    price: float


# This variable is only temporary storage for this small example.
# Restarting the server resets it.
next_product_id = 1


@app.post(
    "/products",
    response_model=ProductResponse,
    status_code=status.HTTP_201_CREATED
)
def create_product(product: ProductCreate):
    global next_product_id

    # The server generates the ID; the client does not provide it.
    new_product = {
        "id": next_product_id,
        "name": product.name,
        "price": product.price
    }

    next_product_id += 1

    # FastAPI checks/serializes the returned data using ProductResponse.
    return new_product


# Run:
# uvicorn api_program_10:app --reload
#
# Try this body in /docs:
# {
#   "name": "Mouse",
#   "price": 500
# }
#
# The response should include an id, name, and price.


## Program 11 — Update only the fields the client provides

For an update, the client might send only a new price and leave the name unchanged.

This example uses `model_dump(exclude_unset=True)`. It returns only the fields the client actually supplied. That is different from dumping every field, because defaults for omitted fields would otherwise appear in the dictionary.

Save as `api_program_11.py` and run:
`uvicorn api_program_11:app --reload`

This example uses a small in-memory dictionary so you can focus on the model behavior.

In [ ]:
# PROGRAM 11: An update model with optional fields

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI()

class ProductUpdate(BaseModel):
    # Every field is optional because an update may change only one thing.
    name: str | None = Field(default=None, min_length=2)
    price: float | None = Field(default=None, gt=0)


# Pretend this dictionary is a tiny database for the example.
products = {
    1: {"id": 1, "name": "Keyboard", "price": 1200.0}
}


@app.put("/products/{product_id}")
def update_product(product_id: int, changes: ProductUpdate):
    # Look up the product using the ID in the URL.
    if product_id not in products:
        raise HTTPException(status_code=404, detail="Product not found")

    # Keep only fields the client sent in the request body.
    supplied_fields = changes.model_dump(exclude_unset=True)

    # Apply each supplied field to the existing product.
    for field_name, value in supplied_fields.items():
        products[product_id][field_name] = value

    return products[product_id]


# Run:
# uvicorn api_program_11:app --reload
#
# In /docs, call PUT /products/1 with:
# {
#   "price": 1300
# }
#
# The name should remain "Keyboard" and the price should become 1300.


## Program 12 — Practice: predict what will happen

Before running each variation, predict the result. Then run it and check.

1. Change the product name to an empty string.
2. Change the price to `0`.
3. Remove the price field.
4. Change the price to the string `"150"`.
5. Change the price to `"not a number"`.

Remember:
- Missing required field → validation error.
- Wrong value that cannot be converted → validation error.
- Compatible value → may be converted.
- Value violating a `Field` rule → validation error.

In [ ]:
# PROGRAM 12: A small practice program

from pydantic import BaseModel, Field, ValidationError

class Product(BaseModel):
    name: str = Field(min_length=2)
    price: float = Field(gt=0)


test_cases = [
    {"name": "Mouse", "price": 500},
    {"name": "", "price": 500},
    {"name": "Mouse", "price": 0},
    {"name": "Mouse", "price": "150"},
    {"name": "Mouse", "price": "not a number"},
]

for number, data in enumerate(test_cases, start=1):
    print(f"\nTest case {number}: {data}")

    try:
        product = Product(**data)
        # **data passes dictionary keys as named arguments.
        print("VALID:", product)

    except ValidationError as error:
        print("INVALID:")
        print(error)


# Quick reference

| Concept | Simple meaning |
|---|---|
| `BaseModel` | Base used to create a Pydantic model |
| `name: str` | A required text field |
| `price: float` | A required numeric field |
| `str | None = None` | A field that may be omitted or set to `None` |
| `Field(gt=0)` | Value must be greater than zero |
| `Field(min_length=2)` | Text must contain at least two characters |
| `ValidationError` | Pydantic's exception for invalid data |
| `model_dump()` | Convert a model to a dictionary |
| `model_dump(exclude_unset=True)` | Include only fields explicitly supplied |
| `response_model=...` | Tell FastAPI what shape the response should have |
| HTTP 422 | FastAPI's usual status for invalid request data |